In [ ]:
import numpy as np
import pandas as pd
import re
import unicodedata
import tensorflow as tf

# Corpus Preprocess Notebook

This file is devoted to transforming the Palaeohispanic data extracted from the Hesperia Database into a dataset useful for Machine Learning projects.

In [2]:
iberian_data = pd.read_csv('./src/iberian_raw.csv',sep=';',keep_default_na=False)
celtiberian_data = pd.read_csv('./src/celtiberian_raw.csv',sep=';',keep_default_na=False)
es_cities_data = pd.read_csv('./src/ES_cities.csv',sep=';',keep_default_na=False,encoding='latin1',decimal=',')
fr_cities_data = pd.read_csv('./src/FR_cities.csv')
provinces_data = pd.read_csv('./src/provinces.csv')

## 1. Location

In [3]:
def remove_diacritics(text):
    DIACRITICS = ['\u0303','\u0301','\u0308','\u0327','\u0300']
    res = re.sub('|'.join(DIACRITICS),'',unicodedata.normalize('NFKD',text))
    OTHER_MARKS = ['-']
    res = re.sub('|'.join(OTHER_MARKS),' ',res)
    APOSTROPHES = ["'",'\x92']
    res = re.sub('|'.join(APOSTROPHES),"'",res)
    return res

def apply_manual_tweaks(text,changes):
    return changes.get(text,text)

### Municipalities

In [4]:
municipalities = pd.DataFrame(np.concatenate([
    fr_cities_data[['label','latitude','longitude','department_name']].to_numpy(),
    es_cities_data[['NOMBRE_ACTUAL','LATITUD_ETRS89','LONGITUD_ETRS89','PROVINCIA']].to_numpy()
    ],axis=0
    ),columns=['name','latitude','longitude','province']
)

In [5]:
LOCATION_MANUAL_CORRECTIONS = {
    'sant boi de llogregat':'sant boi de llobregat',
    'foz calanda':'foz-calanda',
    'la tor de querol':'latour de carol',
    'cerca de mogente ?':'mogente',
    'verdolay, murcia':'murcia',
    'ares del maestre':'ares del maestrat',
    'torres-torres':'torres torres',
    'castellon':'castello de la plana',
    'comarca camp de morvedre':'sagunto',
    'agramon, hellin':'hellin',
    'benlloch|bell lloc':'benlloc',
    'nissan les enserune':'nissan lez enserune',
    'enveig':'enveitg',
    'isona i la conca della':'isona i conca della',
    'comarca del alt emporda':'figueres',
    'nauja':'nahuja',
    'penalba de castro':'huerta de rey',
    'trebago':'trevago',
    'ibiza':'eivissa',
    'el burgo de osma':'burgo de osma',
    'olleros de pisuerga':'aguilar de campoo',
    'padilla de duero penafiel':'penafiel',
    'cubillejo de la sierra':'molina de aragon',
    'taracena':'guadalajara',
    'cantoral de la pena':'castrejon de la pena',
    'villaren de valdivia':'pomar de valdivia',
    'ventosa (san pedro de manrique)':'san pedro manrique'
}

In [6]:
municipalities.loc[:,'name'] = municipalities.loc[:,'name'].str.lower().str.strip()\
    .apply(lambda m : apply_manual_tweaks(remove_diacritics(m),LOCATION_MANUAL_CORRECTIONS))
municipalities.loc[:,'province'] = municipalities.loc[:,'province'].str.lower().str.strip()\
    .apply(lambda m : remove_diacritics(m))

In [7]:
def get_municipalities_coordinates(data,municipalities,manual_corrections):
    municipalities_latitudes = []
    municipalities_longitudes = []

    for i,(m,p) in data[['municipio','provincia']].iterrows():
        this_municipality = apply_manual_tweaks(
            remove_diacritics(re.sub('\s*/\s*','|',m.lower().strip())),
            manual_corrections
        )
        this_province = remove_diacritics(re.sub('\s*/\s*','|',p.lower().strip()))
        
        if re.match('desconocid[oa]',this_municipality) or re.match('desconocid[oa]',this_province):
            municipalities_latitudes.append(None)
            municipalities_longitudes.append(None)
            continue

        matches = municipalities[(municipalities['name'].str.contains(this_municipality))&(municipalities['province'].str.contains(this_province))]
        if len(matches) > 0:
            municipalities_latitudes.append(float(matches['latitude'].values[0]))
            municipalities_longitudes.append(float(matches['longitude'].values[0]))
        else:
            only_municipality_match = matches = municipalities[(municipalities['name'].str.contains(this_municipality))]
            if len(only_municipality_match) > 0:
                municipalities_latitudes.append(float(only_municipality_match['latitude'].values[0]))
                municipalities_longitudes.append(float(only_municipality_match['longitude'].values[0]))
            else:
                print(i,this_municipality,this_province)

    return municipalities_latitudes, municipalities_longitudes

In [8]:
iberian_municipalities_coordinates = get_municipalities_coordinates(iberian_data,municipalities,LOCATION_MANUAL_CORRECTIONS)
iberian_data['municipality_latitude'] = iberian_municipalities_coordinates[0]
iberian_data['municipality_longitude'] = iberian_municipalities_coordinates[1]

In [9]:
celtiberian_municipalities_coordinates = get_municipalities_coordinates(celtiberian_data,municipalities,LOCATION_MANUAL_CORRECTIONS)

celtiberian_data['municipality_latitude'] = celtiberian_municipalities_coordinates[0]
celtiberian_data['municipality_longitude'] = celtiberian_municipalities_coordinates[1]

### Provinces

In [10]:
provinces_data.loc[:,'name'] = provinces_data.loc[:,'name'].str.lower().str.strip().apply(lambda w : remove_diacritics(w))

In [11]:
def get_province_coordinates(province_list,mapping):
    latitudes = []
    longitudes = []
    for province in province_list:
        if province == '' or re.match('desconocid[oa]',province):
            latitudes.append(None)
            longitudes.append(None)
            continue

        match = mapping[mapping['name'] == (province)]
        if len(match) > 0:
            latitudes.append(float(match['lat'].values[0]))
            longitudes.append(float(match['lon'].values[0]))
        else:
            print(province)
            latitudes.append(None)
            longitudes.append(None)
    return latitudes,longitudes

In [12]:
clean_iberian_provinces = iberian_data['provincia'].str.lower().str.strip().apply(lambda w : remove_diacritics(w))
iberian_province_coordinates = get_province_coordinates(clean_iberian_provinces,provinces_data)
iberian_data['province_latitude'] = iberian_province_coordinates[0]
iberian_data['province_longitude'] = iberian_province_coordinates[1]

In [13]:
clean_celtiberian_provinces = celtiberian_data['provincia'].str.lower().str.strip().apply(lambda w : remove_diacritics(w))
celtiberian_province_coordinates = get_province_coordinates(clean_celtiberian_provinces,provinces_data)
celtiberian_data['province_latitude'] = celtiberian_province_coordinates[0]
celtiberian_data['province_longitude'] = celtiberian_province_coordinates[1]

portugal


## 2. Chronology

### Definitions

In [14]:
CENTURY_VALUE = {'I':[1,100],'II':[101,200],'III':[201,300],'IV':[301,400],'V':[401,500]}

def substitute_century(text):
    print(text)
    century = re.search('[Ss][Ii][Gg][Ll][Oo]\s*\.?\s*(.*?)([Aa]\.[EeCc]\.)',text)
    if century:
        century = century.groups()
        print(century)
        isBeforeChrist = re.match('[Aa]\.[CcEe]\.',century[1].strip())
        century_number_value = CENTURY_VALUE[century[0].strip()]
        if isBeforeChrist:
            century_number_value = -1 * century_number_value[::-1]
        return str(century_number_value)
    else:
        return text
    
FUZZY_KEYWORDS = {
    "Comienzos": lambda x : modify_range_by_value(x,0.2),
    "Inicios": lambda x : modify_range_by_value(x,0.2),
    "Finales": lambda x : modify_range_by_value(x,0.2,0.8),
    "Fin\.": lambda x : modify_range_by_value(x,0.2,0.8),
    "Mediados" : lambda x : modify_range_by_value(x,0.5,0.25),
    "meds?\." : lambda x : modify_range_by_value(x,0.5,0.25),
    "Primera mitad": lambda x : modify_range_by_value(x,0.5),
    "1ª mitad": lambda x : modify_range_by_value(x,0.5),
    "Segunda mitad": lambda x : modify_range_by_value(x,0.5,0.5),
    "2ª mitad": lambda x : modify_range_by_value(x,0.5,0.5),
    "Primer tercio": lambda x : modify_range_by_value(x,1/3),
    "Último tercio": lambda x : modify_range_by_value(x,1/3,2/3),
    "Primer cuarto": lambda x : modify_range_by_value(x,0.25),
    "1er ¼": lambda x : modify_range_by_value(x,0.25),
    "Segundo cuarto": lambda x : modify_range_by_value(x,0.25,0.25),
    "Tercer cuarto": lambda x : modify_range_by_value(x,0.25,0.5),
    "Último cuarto": lambda x : modify_range_by_value(x,0.25,0.75),
    "Anterior": lambda x : modify_range_by_before(x),
    "En torno": lambda x : modify_range_center(x,20),
    "Hacia": lambda x : modify_range_center(x,20),
    "Alrededor": lambda x : modify_range_center(x,20),
    "ca\.": lambda x : modify_range_center(x,20),
    "Aprox": lambda x : modify_range_center(x,20)
}

def modify_range_by_value(range,value=1,shift=0):
    max_val = range[1] - range[0]
    total_shift = np.round(max_val * shift)
    max_val = int(max_val * value)
    return [int(range[0] + total_shift),int(range[0] + max_val + total_shift)]

def modify_range_by_before(range):
    return [None,int(range[1])]

def modify_range_center(range,value=0):
    return [int(range[0] - value),int(range[1] + value)]

def apply_modifiers(text_list):
    for i in range(len(text_list)):
        if isinstance(text_list[i],str):
            for keyword in FUZZY_KEYWORDS:
                if re.search(keyword.lower(),text_list[i],re.IGNORECASE):
                    for j in range(i+1,len(text_list)):
                        if isinstance(text_list[j],list) and len(text_list[j]) == 2:
                            text_list[j] = FUZZY_KEYWORDS.get(keyword)(text_list[j])
                            break
    return text_list

def apply_era(text_list):
    last_index = 0
    for i in range(len(text_list)):
        if isinstance(text_list[i],str):
            if re.search('[Aa]\.?\s*[CcEe]\.?',text_list[i]):
                for j in range(last_index,i):
                    if isinstance(text_list[j],list) and len(text_list[j]) == 2:
                        text_list[j] = [-1 * t for t in text_list[j][::-1]]
                    last_index = i
    return text_list

def get_biggest_interval(text_list):
    intervals = np.array([e for e in text_list if isinstance(e,list) and len(e)==2])
    if intervals.shape[0] > 0:
        min0 = None if None in intervals[:,0] else int(intervals[:,0].min())
        return [min0,int(intervals[:,1].max())]
    else:
        return None
    
def change_str_to_interval(text):
    if re.match('\\b(\d+)\\b',text):
        return [int(text),int(text)]
    elif re.search('^era',text,re.IGNORECASE):
        return [1,1]
    else:
        return CENTURY_VALUE.get(text,text)

IBERIAN_MANUAL_TWEAKS = {
    "Siglo IV a.C. (mediados o segunda mitad, probablemente)":modify_range_by_value([-400,-300],0.5,0.25),
    "Simón Cornago (2013, 157): siglo I a.C. Rodríguez Ramos (2004, 207): 200 - 50 a.C.":[-200,-50],
    "Aprox. 275 - 200 a.C.":[-275,-200],
    "Aprox. 75 - 25 a.C., coincidiendo con la época tardorrepublicana.":[-75,-25],
    "Panosa (2001, 524-525): primer cuarto del s. I a.C.":modify_range_by_value([-100,-1],0.25),
    "Anterior a la amortización del silo entre el 80 a.C. y el 40 a.C. Según Velaza (2003, 183) se dataría entre el 125 y el 40 a.C.":[-125,-40],
    "Ferrer i Jané - Escrivà (2014, 212): finales s. III - comienzos del II a.C.":get_biggest_interval([modify_range_by_value([-300,-201],0.2,0.8),modify_range_by_value([-200,-101],0.2)]),
    "Finales s. II - mediados s. I a.C. Noguera (1994, 126): principios - mediados s. I a.C.":get_biggest_interval([modify_range_by_value([-200,-101],0.2,0.8),modify_range_by_value([-100,-1],0.5,0.25)]),
    "J. de Hoz: siglo V - comienzos s. IV a.C. Apareció reutilizada en un contexto de los siglos III y II a.C., lo que constituye una fecha ante quem.":get_biggest_interval([[-500,-401],modify_range_by_value([-400,-301],0.2)]),
    "Anterior al II a.C. (Llobregat). Ramos Fernández 1969: anterior al III aC. y posterior al VI a.C.":get_biggest_interval([[-500,-401],[-400,-301]]),
    "Siglos IV - III a.C. F. Quesada (1997, 123), considera que no debe ser anterior al siglo III a.C.":get_biggest_interval([[-400,-301],[-300,-201]]),
    "Rodríguez Ramos: principios s. II a.C.; Fletcher y Mesado (1968, 30): ss. II-I a.C; Fletcher (1967, 7): s. II a.C.":get_biggest_interval([[-200,-101],[-100,-1]]),
    "Finales s. III a.C. - comienzos s. II a.C. Fletcher - Mesado (1968, 30): ss. II - I a.C.":get_biggest_interval([modify_range_by_value([-300,-201],0.2,0.8),modify_range_by_value([-200,-101],0.2),[-200,-101],[-101,-1]]),
    "Rodríguez Ramos (2004, 220): 200 - 50 a.C.":[-200,-50],
    "Bonet (1995, 405): siglos III - II a.C. Rodríguez Ramos (2004, 218): 300 - 175 a.C.":[-300,-175],
    "Posterior al siglo IV a.C. Rodríguez Ramos: 210-180 a.C.":[-210,-180],
    "No hay criterios para datar la inscripción, aunque el uso de be5 es fechado por Rodríguez Ramos (2004, 113), a finales del siglo III y comienzos del II a.C.":get_biggest_interval([modify_range_by_value([-300,-201],0.2,0.8),modify_range_by_value([-200,-101],0.2)]),
    "II/I a. C. Producto provincial tardío según Barberá (1975).":get_biggest_interval([[-200,-101],[-100,-1]]),
    "Siglo I a.C. Rodríguez Ramos (2004, 22)1: 150 - 50 a.C.":[-150,-50],
    "La estela y la inscripción latina se fechan a finales del siglo I o comienzos del II d.C. (Corell 1989; Corell 1996; Martínez Valle 1993, 249). Esta datación ofrece la fecha post quem para el esgrafiado ibérico, que debe ser posterior.":get_biggest_interval([modify_range_by_value([1,100],0.2,0.8),modify_range_by_value([101,200],0.2)]),
    "Finales s. IV a.C. Sarrión (1981): ca. 325 a.C.; Rodríguez Ramos (2004): 325-300 a.C.":[-325,-300],
    "Tipología cerámica: fiinales s. III - s. II a.C.. Rodríguez Ramos: 250-180 a.C.":[-250,-180],
    "Siglos II - I a.C. Moncunill (2007, 126) la data en el siglo I a.C.":get_biggest_interval([[-200,-101],[-100,-1]]),
    "350-325 a.C. (CI, nº 92). Jannoray lo databa en el s. II a. C.":get_biggest_interval([[-350,-325],[-200,-101]]),
    "-100/-50":[-100,-50],
    "Incierta: ¿siglo IV a.C.? (Cura 1986, 207, fig. 4.1)":[-400,-301],
    "Incierta: s. II a.E. en adelante.":[-200,None],
    "Primera mitad del s. I a.C. Panosa (2015, 120) afina la cronología al segundo cuarto del siglo I a.C.":modify_range_by_value([-100,-1],0.25,0.25),
    "En torno a finales del siglo II a.C. y comienzos del primero. Corresponde al último período de vida del poblado.":get_biggest_interval([modify_range_by_value([-200,-101],0.2,0.8),modify_range_by_value([-100,-1],0.2)]),
    "100aC":[-100,-100],
    "Siglo IV - cambio de Era":[-400,1],
    "Siglo IV - cambio de Era.":[-400,1],
    "Insegura: siglo IV - cambio de era.":[-400,1],
    "Insegura: siglo II - cambio de era.":[-400,1],
    "Insegura: siglos IV - cambio de era.":[-400,1],
    "Insegura: s. IV - cambio de era.":[-400,1],
    "Siglo IV - cambio de era.":[-400,1],
    "S. II-I a. C (-200 a -30).":[-200,-30],
}

CELTIBERIAN_MANUAL_TWEAKS = {
    "Finales del s. III - cambio de Era.":[FUZZY_KEYWORDS['Finales']([-300,-201])[0],-1],
    "Finales del siglo III a.C. hasta, aproximadamente, el cambio de la era.":[FUZZY_KEYWORDS['Finales']([-300,-201])[0],-1],
    "Desde mediados del siglo II hasta el 76 a. e. Esta horquilla temporal comienza en el momento de la reubicación de la población indígena desde el cerro de Monte Cantabria, por imposición romana, tras la primera guerra celtibérica. El final del lapso temporal viene marcado por la violenta destrucción de la ciudad por parte de Sertorio en el 76 a.e.":[-175,-76],
    "I a.C. (avanzado), posiblemente entre c. 30 y 14 a.e.":FUZZY_KEYWORDS['Finales']([-100,-1]),
    "Iglesias y Ruiz 1998, 7475 a.C., tomando como fecha post quem el 16 a.C. sitúan la inscripción entre esa fecha y el 200. Quizá sea recomendable rebajar esta última por motivos lingüísticos, pues se hace difícil pensar, de momento, en la pervivencia de la lengua celtibérica en el s. II, según indica Simón.":[-16,200],
    "Wattenberg (1963, 208) la data entre el 75 a.C. y el 29 a.C.; Arlegui (1992b, 476477 a.C.) a principios del I.":[-75,29],
    "Antes de c. 70 a.C.":FUZZY_KEYWORDS['Anterior']([-70,-70]),
    "Antes del 30 a.C..":FUZZY_KEYWORDS['Anterior']([-30,-30]),
    "S. I a.C. o comienzos del I.":get_biggest_interval([[-100,-1],FUZZY_KEYWORDS['Comienzos']([1,100])]),
}

### Applications

Iberian

In [15]:
iberian_chronology = iberian_data['datacion'].apply(lambda w : [t.strip() for sw in re.split('-|\s+y\s+|\s+al\s+|\s+d?el\s+|/',w) for t in re.split('\s+([AaDd]\.?\s*[CcEe]\.?)',sw) if t != '']) \
    .apply(lambda w : [t.strip() for ws in w for t in re.split('[Ss][Ii][Gg][Ll][Oo][Ss]?\s*([A-Z]+).*',ws) if t != '']) \
    .apply(lambda w : [t.strip() for ws in w for t in re.split('[Ss]\s*\.\s*([A-Z]+).*',ws) if t != '']) \
    .apply(lambda w : [t.strip() for ws in w for t in re.split('\\b(\d+)\\b',ws) if t != '']) \
    .apply(lambda w : [change_str_to_interval(ws) for ws in w]) \
    .apply(lambda w : apply_era(w)) \
    .apply(lambda w : apply_modifiers(w)) \
    .apply(lambda w : get_biggest_interval(w))

for dating in IBERIAN_MANUAL_TWEAKS:
    rows = iberian_chronology.loc[iberian_data['datacion']==dating]
    iberian_chronology.loc[iberian_data['datacion']==dating] = [IBERIAN_MANUAL_TWEAKS[dating]] * len(rows)

I'm leaving null values since the fill can vary depending on the rows chosen

In [16]:
iberian_chronology_min = iberian_chronology.apply(lambda c : c[0] if c else None)
# iberian_chronology_min = iberian_chronology_min.fillna(iberian_chronology_min.min())
iberian_chronology_max = iberian_chronology.apply(lambda c : c[1] if c else None)
# iberian_chronology_max = iberian_chronology_max.fillna(iberian_chronology_max.max())
iberian_chronology_mean = np.mean([iberian_chronology_min,iberian_chronology_max],axis=0)
iberian_chronology_width = iberian_chronology_max - iberian_chronology_min

In [17]:
iberian_data['datacion_min'] = iberian_chronology_min
iberian_data['datacion_max'] = iberian_chronology_max
iberian_data['datacion_mean'] = iberian_chronology_mean
iberian_data['datacion_width'] = iberian_chronology_width

Celtiberian

In [18]:
# First, replace -I, -II, -III with I, II, III a.C. (B.C.)
celtiberian_data['datacion'] = celtiberian_data['datacion'].apply(lambda w : re.sub('-\s*(I{1,3})','\\1 a.C.',w))
celtiberian_data['datacion'] = celtiberian_data['datacion'].apply(lambda w : re.sub('-\s*(\d{1,3})','\\1 a.C.',w))

celtiberian_chronology = celtiberian_data['datacion'].apply(lambda w : [t.strip() for sw in re.split('-|\s+y\s+|\s+al\s+|\s+d?el\s+|/',w) for t in re.split('\s+([AaDd]\.?\s*[CcEe]\.?)',sw) if t != '']) \
    .apply(lambda w : [t.strip() for ws in w for t in re.split('[Ss][Ii][Gg][Ll][Oo][Ss]?\s*([A-Z]+).*',ws) if t != '']) \
    .apply(lambda w : [t.strip() for ws in w for t in re.split('[Ss]\s*\.\s*([A-Z]+).*',ws) if t != '']) \
    .apply(lambda w : [t.strip() for ws in w for t in re.split('\\b(\d+)\\b',ws) if t != '']) \
    .apply(lambda w : [change_str_to_interval(ws) for ws in w]) \
    .apply(lambda w : apply_era(w)) \
    .apply(lambda w : apply_modifiers(w)) \
    .apply(lambda w : get_biggest_interval(w))

for dating in CELTIBERIAN_MANUAL_TWEAKS:
    rows = celtiberian_chronology.loc[celtiberian_data['datacion']==dating]
    celtiberian_chronology.loc[celtiberian_data['datacion']==dating] = [CELTIBERIAN_MANUAL_TWEAKS[dating]] * len(rows)

In [19]:
celtiberian_chronology_min = celtiberian_chronology.apply(lambda c : c[0] if c else None)
# celtiberian_chronology_min = celtiberian_chronology_min.fillna(celtiberian_chronology_min.min())
celtiberian_chronology_max = celtiberian_chronology.apply(lambda c : c[1] if c else None)
# celtiberian_chronology_max = celtiberian_chronology_max.fillna(celtiberian_chronology_max.max())
celtiberian_chronology_mean = np.mean([celtiberian_chronology_min,celtiberian_chronology_max],axis=0)
celtiberian_chronology_width = celtiberian_chronology_max - celtiberian_chronology_min

In [20]:
celtiberian_data['datacion_min'] = celtiberian_chronology_min
celtiberian_data['datacion_max'] = celtiberian_chronology_max
celtiberian_data['datacion_mean'] = celtiberian_chronology_mean
celtiberian_data['datacion_width'] = celtiberian_chronology_width

## 3. Text

From now on, both Iberian and Celtiberian datasets can be merged and worked on at the same time

In [21]:
data = pd.concat([iberian_data,celtiberian_data],axis=0)
data['clean_text'] = data['texto'].copy()

### 1. Replace intervals with with [---]

It means an unknown number of unknown characters

In [22]:
data['clean_text'] = data['clean_text'].apply(lambda w : re.sub('\[[^\[\]]*?c\.?.*?\]','[---]',w))
data['clean_text'] = data['clean_text'].apply(lambda w : re.sub('\[-?\d/\d-?\]','[---]',w))
data['clean_text'] = data['clean_text'].apply(lambda w : re.sub('\[\d-\d\]','[---]',w))
data['clean_text'] = data['clean_text'].apply(lambda w : re.sub('\[\d+\]','[---]',w))

### 2. Remove embedded bibliography and parentheses

Which ones are there?

In [23]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\([^\(\)]+?\)',w)))\
                .values))

{'(tatuŕ)', '(o)', '(2)', '(tontunos?)', '(5)', '(e)', '(vacat)', '(-)', '(texto superior)', '(10)', '(tuno?)', '(8)', '(ta)', '(+)', '(izquierda)', '(u)', '(marca)', '(figura)', '([-])', '(texto inferior, palimpsesto)', '(unos)', '(d)', '(sḿ)', '(?)', '(ntis)', '(koŕ)', '(c)', '(derecha)', '([)', '(s)', '([--])', '(])', '(---)', '(vac)', '(A)', '(tu)', '(irkaiska)', '(tunos)', '(3 bi)', '(nteis?)', '(a)', '(sic ?)', '(sic)', '(nos?)', '(7)', '(1)', '(B)', '(tontuno?)', '(ŕ)', '(os)', '(ENTIS)', '(3)', '(aspa)', '(b)', '(obadil)', '(ba)', '(rbaebaseŕ+er)', '(c2)', '(VM)', '(marca:ϡ?)', '(2015)', '(6)'}


In [24]:
data[data['clean_text'].str.contains('c2')]

,yacimiento,refMLH,refHesperia,texto,municipio,provincia,material,soporte,direccion_escritura,tecnica,...,datacion,municipality_latitude,municipality_longitude,province_latitude,province_longitude,datacion_min,datacion_max,datacion_mean,datacion_width,clean_text
501,El Tossal de Sant Miquel,F.13.12,V.06.017,(a) eŕieun[ (b) ne:benebedaner:iums[ (c1) ]ane...,Liria / Llíria,Valencia,CERAMICA,RECIPIENTE,DEXTROGIRA,PINTADO,...,Rodríguez Ramos: 250-200 a.C. Tipología cerámi...,39.625864,-0.594428,39.48,-0.38,-400.0,-1.0,-200.5,399.0,(a) eŕieun[ (b) ne:benebedaner:iums[---]ane (c...


In [25]:
PARENTHESES_PATTERNS = {
    '(texto inferior, palimpsesto)':'',
    '(vacat)':'',
    '(derecha)':'',
    '(figura)':'',
    '(sic ?)':'',
    '(izquierda)':'',
    '(aspa)':'+',
    '(marca:ϡ\?)':'+',
    'Lectura de Panosa (2015):':'',
    '(marca)':'+',
    '(texto superior)':'+',
    '(sic)':'',
    '(vac)':'',
    '(3 bi)':'',
    '(])':']',
    '(+)':'+',
    '([)':'[',
    '(?)':'?',
    '(-)':'-'
}

for pp,pr in PARENTHESES_PATTERNS.items():
    data['clean_text'] = data['clean_text'].str.replace(re.escape(pp),pr,regex=True)

In [26]:
# Check again
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\([^\(\)]+?\)',w)))\
                .values))

{'(tatuŕ)', '(o)', '(2)', '(tontunos?)', '(5)', '(e)', '(tuno?)', '(10)', '(8)', '(ta)', '(u)', '([-])', '(unos)', '(d)', '(sḿ)', '(ntis)', '(koŕ)', '(c)', '(---)', '(s)', '([--])', '(A)', '(tu)', '(irkaiska)', '(tunos)', '(nteis?)', '(a)', '(nos?)', '(7)', '(1)', '(B)', '(tontuno?)', '(ŕ)', '(os)', '(ENTIS)', '(3)', '(obadil)', '(b)', '(ba)', '(c2)', '(rbaebaseŕ+er)', '(VM)', '(marca:ϡ?)', '(6)'}


More than one character (no numbers) is considered an abbreviation and thus the parentheses are removed

In [27]:
data['clean_text'] = data['clean_text'].str.replace('\(([^\(\)\d]{2,})\)',r'\1',regex=True)

In [28]:
# Check again
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\([^\(\)]+?\)',w)))\
                .values))

{'(c)', '(s)', '(o)', '(2)', '(A)', '(5)', '(e)', '(a)', '(7)', '(1)', '(10)', '(B)', '(8)', '(ŕ)', '(u)', '(3)', '(b)', '(c2)', '(d)', '(6)'}


Only some of these parenthesis correspond to listing items

In [29]:
for token in set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\([^\(\)]+?\)',w)))\
                .values):
    print(token + '\n===')
    for line in data[data['clean_text'].str.contains(re.escape(token))]['clean_text'].values:
        print('***' + line)
    print('===\n===')

(c)
===
***A: iunstir:beleśaiŕ:kaŕkośkar:bastaibaitieba:balkélakośka:bitetui:bar+ iunstir:ekiartone:beleśtar:senḿrun:etesilir:iunstir:etetur[ B: śaner:buranalir:bitan:a+ vacat tauti+[ vacat biten[ (c) balkélakú
***1. ]+baserte:bonantite:nḿbaŕde:bortebara:kaŕesirdeegiar:banite:kaŕ[ 2. ]+irden[ 3. ]+tin+[ 4. ]boi:ban[ 5. (a) egiar (b) ḵa̱ŕestabikiŕ. 6. ebiŕteekiar 7. elolekaŕko 8. (a) oŕodis:(b) kaŕbi 9. beber 10. belar:ban:iŕ. 11. (a) ban (b) iŕ 12. (a) elbebebebeber (b) tibiserbaśdibaa 13. bankuturiŕaker 14. uŕkebas[ 15. daŕa[ 16. (a) ]+banḿibae:(b) ]śai (c) ]+ntautin
***(a) ]ḵo̱[--]s (b) ]tuśerti (c) ai bas kuegiar[ (d) gemiegiar (e) tadaŕ
***A (a) bilosṯe̱ keŕana (b) ta ? B (a) ban (b) a (c) ta ?
***(a) ]b̲a̲in̲a̲karkaśar (b) ]+e (c) ++ ir
***(a) eg̱e̱ŕśore (b) eg̱e̱ŕśore (c) e̱g̱e̱ṟ́ś̱o̱ṟe̱
***(a) kambarokum (b) bak̲a̲ (c) kai̲ (d) l̲ (e) l̲
===
===
(s)
===
***A: 1a gau(ŕ)ubastigi 1b gaikeirkaiskarta(u)tinenobadil 2 otanar :eŕebauśidirteśierban(e)sidar 3 kaisurarbidan:sakari(s)k

In [30]:
data['clean_text'] = data['clean_text'].str.replace('[^ ]+\(([^\(\)\d]+?)\)[^ ]+',r'\1',regex=True)

In [31]:
# Check again
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\([^\(\)]+?\)',w)))\
                .values))

{'(c)', '(2)', '(A)', '(5)', '(e)', '(a)', '(7)', '(1)', '(10)', '(B)', '(8)', '(3)', '(b)', '(c2)', '(d)', '(6)'}


In [32]:
for token in set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\([^\(\)]+?\)',w)))\
                .values):
    print(token + '\n===')
    for line in data[data['clean_text'].str.contains(re.escape(token))]['clean_text'].values:
        print('***' + line)
    print('===\n===')

(c)
===
***A: iunstir:beleśaiŕ:kaŕkośkar:bastaibaitieba:balkélakośka:bitetui:bar+ iunstir:ekiartone:beleśtar:senḿrun:etesilir:iunstir:etetur[ B: śaner:buranalir:bitan:a+ vacat tauti+[ vacat biten[ (c) balkélakú
***1. ]+baserte:bonantite:nḿbaŕde:bortebara:kaŕesirdeegiar:banite:kaŕ[ 2. ]+irden[ 3. ]+tin+[ 4. ]boi:ban[ 5. (a) egiar (b) ḵa̱ŕestabikiŕ. 6. ebiŕteekiar 7. elolekaŕko 8. (a) oŕodis:(b) kaŕbi 9. beber 10. belar:ban:iŕ. 11. (a) ban (b) iŕ 12. (a) elbebebebeber (b) tibiserbaśdibaa 13. bankuturiŕaker 14. uŕkebas[ 15. daŕa[ 16. (a) ]+banḿibae:(b) ]śai (c) ]+ntautin
***(a) ]ḵo̱[--]s (b) ]tuśerti (c) ai bas kuegiar[ (d) gemiegiar (e) tadaŕ
***A (a) bilosṯe̱ keŕana (b) ta ? B (a) ban (b) a (c) ta ?
***(a) ]b̲a̲in̲a̲karkaśar (b) ]+e (c) ++ ir
***(a) eg̱e̱ŕśore (b) eg̱e̱ŕśore (c) e̱g̱e̱ṟ́ś̱o̱ṟe̱
***(a) kambarokum (b) bak̲a̲ (c) kai̲ (d) l̲ (e) l̲
===
===
(2)
===
***A: (a): ]śkilir:uduta:baśir:taŕakaŕ ]nS45 (b): otalauS45S48der:sikilS48ŕikan etaS81rkir:sosintikerka:nanban bankiśaŕikan

### 3. Brackets

In [33]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\[[^\[\]]+?\]',w)))\
                .values))

{'[ 4. ]', '[ ]', '[..]', '[------]', '[ ?]', '[-]', '[bakaś]', '[│?]', '[-1-2-]', '[ A2a2 ]', '[ // ]', '[/ kaneka:śalir:ka IIIIIIIIII b̲a̲:iuni̱bilose B: ]', '[ B2 ]', '[ḵu̱+++]', '[tiri]', '[--/]', '[te]', '[ń]', '[VS, -A ---]', '[---?]', '[ 3 ]', '[++]', '[ 5 ]', '[an]', '[ _______ ]', '[tul]', '[ A2b2 ]', '[ ++rkaliṟa AII: +a̱rṯa̲+ B: ]', '[ - - ]', '[?]', '[̱---]', '[–]', '[ar]', '[ŕ?]', '[-2-3-]', '[s]', '[+]', '[ B: ]', '[u]', '[-?]', '[ (b) ]', '[ --- ]', '[ka]', '[I]', '[ o ]', '[m ?]', '[ś]', '[śale]', '[leitaŕtin]', '[be]', '[-----]', '[--- śal]', '[ v iniltiŕe: ]', '[--- s]', '[IIII]', '[tir+ i]', '[.?]', '[ś ---]', '[ sabakiti ]', '[ 3. ]', '[…]', '[anbi]', '[G]', '[...?]', '[ 2. ]', '[...]', '[ eteso - - -]', '[ +]', '[ba]', '[r-]', '[--- i]', '[ti]', '[--------]', '[ 16. (a) ]', '[-1-2?-]', '[ b) ]', '[ A2b1 ]', '[–––]', '[---]', '[ - - - ]', '[ ------------------ Fragm. b): ]', '[-3?-]', '[ . ]', '[‐‐‐]', '[lt;]', '[ltun]', '[a]', '[ - - -]', '[3?]', '[.]', '[? ]', 

In [34]:
BRACKETS_PATTERNS1 = {
    '[--- śal]':'[---] śal',    '[v]':'',
    '[ v iniltiŕe: ]':'[ iniltiŕe: ]',
    '[ś ---]':'ś [---]',    '[ o ]':' o ',
    '[ń]':'ń',    '[--- i]':'[---] i',
    '[ 3 ]':'[---]', # Unsure
    '[ŕtin]':'ŕtin',    '[an]':'an',
    '[m]':'m',    '[ŕ?]':'ŕ',
    '[--- s]':'[---] s',    '[ś ---]':'ś [---]',
    '[be]':'be',    '[ 5 ]':'[ ]',
    '[n]':'n',    '[ei]':'ei',
    '[di]':'di',    '[ // ]':'[ ]',
    '[ne]':'ne',    '[ar]':'ar',
    '[ _______ ]':'[ ]',    '[a]':'a',
    '[o]':'o',    '[bakaś]':'bakaś',
    '[ltun]':'ltun',    '[e]':'e',
    '[lt;]':'lt;',    '[tiri]':'tiri',
    '[VS, -A ---]':'VS, +A [---]',
    '[leitaŕtin]':'leitaŕtin',
    '[r-]':'r+',    '[sa]':'sa',
    '[u]':'u',    '[m ?]':'m+',
    '[ŕte]':'ŕte',    '[tul]':'tul',
    '[tir+ i]':'tir+ i',    '[IIII]':'IIII',
    '[:]':':',    '[s]':'s',
    '[--- śal]':'[---] śal',    '[i]':'i',
    '[ti]':'ti',    '[ba]':'ba',
    '[ne:]':'ne:',    '[ka]':'ka',
    '[ś]':'ś',    '[I]':'I',
    '[ eteso - - -]':'[ eteso [---]',
    '[anbi]':'anbi',     '[te]':'te',
    '[ḵu̱+++]':'ḵu̱+++',    '[śale]':'śale',
    '[G]':'G',    '[ 5]':''
}

BRACKETS_PATTERNS2 = {
    '|'.join([re.escape(v) for v in ['[.?]','[│?]','[+]','[ ?]',
              '[-1-]','[-?]','[–]','[-]',
              '[ - ]','[ . ]','[?]','[.]']]):'+',
    '|'.join([re.escape(v) for v in ['[̱---]','[++]','[...?]','[- -]','[̣--]',
              '[-1-2-]','[...]','[--','-]','[ --- ]',
              '[ - - - ]','[̣̣2]','[---?]','[–––]',
              '[--------]','[ - - -]','[-----]',
              '[ - - ]','[…]','[-2-3-]','[-3?-]',
              '[----]','[..]','[------]','[--]',
              '[‐‐‐]','[---]','[...?]',
              '[-1-2?-]','[-2-]','[3?]',
              '[--/]']]):'[---]',
}

In [35]:
for pp,pr in BRACKETS_PATTERNS1.items():
    data['clean_text'] = data['clean_text'].str.replace(re.escape(pp),pr,regex=True)

In [36]:
for pp,pr in BRACKETS_PATTERNS2.items():
    data['clean_text'] = data['clean_text'].str.replace(pp,pr,regex=True)

In [37]:
data['clean_text'] = data['clean_text'].str.replace('\[---\]\[---\]','[---]',regex=True)

In [38]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\[[^\[\]]+?\]',w)))\
                .values))

{'[ 4. ]', '[? ]', '[ ]', '[ A2b2 ]', '[ sabakiti ]', '[ 3. ]', '[ ++rkaliṟa AII: +a̱rṯa̲+ B: ]', '[ 2. ]', '[ +]', '[ B: ]', '[ 16. (a) ]', '[ A2a2 ]', '[ (b) ]', '[ b) ]', '[ A2b1 ]', '[/ kaneka:śalir:ka IIIIIIIIII b̲a̲:iuni̱bilose B: ]', '[---]', '[ B2 ]', '[ ------------------ Fragm. b): ]', '[ A1b ]', '[ iniltiŕe: ]', '[ nl:IIII I:IIII:baiseltúnebaśiren baśurbiśisa :II 5 AII: ]'}


#### Curly brackets

They are considered errors, and thus are removed

In [39]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\{[^\{\}]+?\}',w)))\
                .values))

{'{kili}', '{dage}'}


In [40]:
for pp in set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\{[^\{\}]+?\}',w)))\
                .values):
    data['clean_text'] = data['clean_text'].str.replace(re.escape(pp),'',regex=True)

In [41]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall('\{[^\{\}]+?\}',w)))\
                .values))

set()


### 4. Numbers

These pattern detects line counters

In [42]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'[\s\b]\d+[\s\b]',w)))\
                .values))

{' 1 ', ' 2 ', ' 12 ', ' 9 ', ' 18 ', ' 15 ', ' 10 ', ' 24 ', ' 21 ', ' 3 ', ' 4 ', ' 5 ', ' 6 '}


In [43]:
for pp in set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'[\s\b]\d+[\s\b]',w)))\
                .values):
    data['clean_text'] = data['clean_text'].str.replace(pp,' ',regex=True)

In [44]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'[\s\b]\d+[\s\b]',w)))\
                .values))

set()


This pattern is found in one inscription and it only marks every line

In [45]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'\s\d[ab]?\s',w)))\
                .values))

{' 3b ', ' 1b ', ' 1a ', ' 3a '}


In [46]:
for pp in set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'\s\d[ab]?\s',w)))\
                .values):
    data['clean_text'] = data['clean_text'].str.replace(pp,' ',regex=True)

In [47]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'\s\d[ab]?\s',w)))\
                .values))

set()


In [48]:
def set_of_matches(pattern,text):
    return set.union(*text \
                .apply(lambda w : set(re.findall(pattern,w)))\
                .values)

def substitute_patterns_in_text(text,patterns,substitution=''):
    for pattern in patterns:
        text = text.str.replace(pattern,substitution,regex=True)

### 5. Disputed characters

In [49]:
data['clean_text'] = data['clean_text'].str.replace('⌶|∑|𐘃|‡|ϡ','+',regex=True)

Graphemes without transcription but with codes:

- SXY
- .SXYf.

In [50]:
print(list(sorted(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'\s*\.?S\d{2,}\.?f?\.?\s*',w)))\
                .values),
                reverse=True,
                key=lambda w : len(w))))

[' .S47f. ', '.S58. ', ' S46 ', ' S82', 'S45 ', 'S48 ', 'S47f', 'S45', 'S48', 'S81']


In [51]:
for pp in list(sorted(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'\s*\.?S\d{2,}\.?f?\.?\s*',w)))\
                .values),
                reverse=True,
                key=lambda w : len(w))):
    data['clean_text'] = data['clean_text'].str.replace(re.escape(pp),'+',regex=True)

In [52]:
print(set.union(*data['clean_text'] \
                .apply(lambda w : set(re.findall(r'\.?S\d{2,}\.?f?\.?',w)))\
                .values))

set()


### 6. Other characters

In [53]:
print(set_of_matches(r'(?<![SABab\(-])\d+?(?![\d\.\):])',data['clean_text']))

{'1', '10', '5', '6'}


In [54]:
data['clean_text'] = data['clean_text'].str.replace(r'(?<![ABab\(-])\d+?(?![\d\.\):])','',regex=True)

In [55]:
print(set_of_matches(r'(?<![ABab\(-])\d+?(?![\d\.\):])',data['clean_text']))

set()


Metrology:

In [56]:
data['clean_text'] = data['clean_text'].str.replace('lt;','<',regex=True)
data['clean_text'] = data['clean_text'].str.replace('gt;','>',regex=True)
data['clean_text'] = data['clean_text'].str.replace('Ι','I',regex=True)

In [57]:
data['clean_text'] = data['clean_text'].str.replace('…','...',regex=True)
data['clean_text'] = data['clean_text'].str.replace(',','',regex=True)
data['clean_text'] = data['clean_text'].str.replace('\|','',regex=True)
data['clean_text'] = data['clean_text'].str.replace('\│','',regex=True)
data['clean_text'] = data['clean_text'].str.replace('/',' ',regex=True)
data.loc[data['refHesperia']=='SP.02.16','clean_text'] = data.loc[data['refHesperia']=='SP.02.16','clean_text'].str.replace('T','+',regex=True)


These are not listings but lines "numbered" with capital letters

In [58]:
data.loc[data['refHesperia']=='SP.02.06','clean_text'] = data[data['refHesperia']=='SP.02.06']['clean_text'].str.replace('[A-Z]:\s','',regex=True)
data.loc[data['refHesperia']=='SP.02.01','clean_text'] = data[data['refHesperia']=='SP.02.01']['clean_text'].str.replace('[A-Z]:\s','',regex=True)
data.loc[data['refHesperia']=='VA.02.01','clean_text'] = data[data['refHesperia']=='VA.02.01']['clean_text'].str.replace('<ba>','ba',regex=True)

Extra annotation elements:

In [59]:
EXTRA_PATTERNS = {
    'Cara':'',
    'externa:?':'',
    'interior:?':'',
    re.escape('<AQUÍ UN SIGNO>'):'+',
    'vacat':' ',
    'Título\s*:?':'',
    'Línea':'',
    'invertida:?':'',
    'lado':'',
    'izquierdo':'',
    'derecho':'',
    'Fragm.':'',
    'Lectura de':'',
    'la editio':'',
    'princeps':'',
    'Líneas':'',
    'superiores':'',
    'Columna':'',
    'marca':'+'
}

In [60]:
for pp,pr in EXTRA_PATTERNS.items():
    data['clean_text'] = data['clean_text'].str.replace(pp,pr,regex=True)

### 7. Words separated in different lines

It seems that words separated in several lines are indicated with a "-" at the end of the line.

In [61]:
data['clean_text'] = data['clean_text'].str.replace('([^-])-\s',r'\1',regex=True)

### 8. Redundancy

Some inscriptions present vowel redundancy; that is, after a syllable, the same vowel is repeated to clarify the vowel sound of the syllable grapheme, but it does not represent a long vowel. There are specific cases.

In [62]:
HESPERIA_ID_REDUNDANT = ['B.01.06','BU.01.01','BU.03.01','BU.06.01',
                         'BU.06.02','NA.07.02','BU.06.05','BU.06.04']
for i in HESPERIA_ID_REDUNDANT:
    data.loc[data['refHesperia']==i,'clean_text'] = data.loc[data['refHesperia']==i,'clean_text'].str.replace(r'([kgtdb]a)a',r'\1',regex=True)
    data.loc[data['refHesperia']==i,'clean_text'] = data.loc[data['refHesperia']==i,'clean_text'].str.replace(r'([kgtdb]e)e',r'\1',regex=True)
    data.loc[data['refHesperia']==i,'clean_text'] = data.loc[data['refHesperia']==i,'clean_text'].str.replace(r'([kgtdb]i)i',r'\1',regex=True)
    data.loc[data['refHesperia']==i,'clean_text'] = data.loc[data['refHesperia']==i,'clean_text'].str.replace(r'([kgtdb]o)o',r'\1',regex=True)
    data.loc[data['refHesperia']==i,'clean_text'] = data.loc[data['refHesperia']==i,'clean_text'].str.replace(r'([kgtdb]u)u',r'\1',regex=True)

In [63]:
# for p in data[data['clean_text'].str.contains(r'[kgtd]aa|[kgtd]ee|[kgtd]ii|[kgtd]oo|[kgtd]uu')][['refHesperia','clean_text']].values:
# for p in data[data['clean_text'].str.contains(r'(?<=[\b\w])[A-Z]:\s')][['refHesperia','clean_text']].values:
    # print('# ' + ' ### '.join(p))

### 9. Last steps for text

In [64]:
data['clean_text'] = data['clean_text'].str.replace('\?','',regex=True)
data['clean_text'] = data['clean_text'].str.replace('-{4,}','',regex=True)

There is no explanation for this J on the database, so I will leave out this instance. It has to do with metrology since it is next to PI, but there are no other instances where this J appears.

In [65]:
data = data[data['refHesperia']!='HGA.01.06']

In [66]:
data = data[data['clean_text']!='']
data['clean_text'] = data['clean_text'].apply(lambda w : unicodedata.normalize('NFKD',w))

In [67]:
print(list(sorted(data['clean_text'].str.split('').explode().value_counts().index.values)))

['', ' ', '(', ')', '*', '+', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', '<', '=', '>', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'X', 'Y', '[', ']', 'a', 'b', 'c', 'd', 'e', 'g', 'i', 'k', 'l', 'm', 'n', 'o', 'r', 's', 't', 'u', 'v', 'z', '·', '́', '̂', '̄', '̇', '̌', '̠', '̣', '̱', '̲', '̵', 'Α', 'Δ', 'Ε', 'Κ', 'Ν', 'Π', 'Ω', '∙']


## 4. Other attributes

The rest of the attributes that can be categorised and mapped into integers will be transformed so that the most common value is assigned 1, the second 2, and so on, while 0 will be reserved for out-of-vocabulary categories.

In [68]:
data.columns

Index(['yacimiento', 'refMLH', 'refHesperia', 'texto', 'municipio',
       'provincia', 'material', 'soporte', 'direccion_escritura', 'tecnica',
       'signario', 'sistema_dual', 'separadores', 'datacion',
       'municipality_latitude', 'municipality_longitude', 'province_latitude',
       'province_longitude', 'datacion_min', 'datacion_max', 'datacion_mean',
       'datacion_width', 'clean_text'],
      dtype='object')

### Material

In [69]:
data['material'].value_counts(),data['material'].value_counts().index.shape

(material
 CERAMICA          1251
 PIEDRA             240
 PLOMO              146
 BRONCE              81
 PLATA               22
                      7
 HUESO                3
 HIERRO               2
 ALABASTRO            1
 ASTA DE CIERVO       1
 opus signinum        1
 Opus signinum        1
 latón                1
 Name: count, dtype: int64,
 (13,))

There are two classes called the same way but different in the first "O".

In [70]:
data['material_cat'] = data['material'].str.upper()
data['material_cat'].value_counts(),data['material_cat'].value_counts().index.shape

(material_cat
 CERAMICA          1251
 PIEDRA             240
 PLOMO              146
 BRONCE              81
 PLATA               22
                      7
 HUESO                3
 HIERRO               2
 OPUS SIGNINUM        2
 ALABASTRO            1
 ASTA DE CIERVO       1
 LATÓN                1
 Name: count, dtype: int64,
 (12,))

In [71]:
material_categoriser = tf.keras.layers.StringLookup()
material_categoriser.adapt(data['material_cat'])
data['material_cat_code'] = material_categoriser(data['material_cat'])

### Medium (soporte)

Same lower/uppercase problem

In [72]:
data['soporte'].value_counts(),data['soporte'].value_counts().shape

(soporte
 RECIPIENTE                          1179
 LAMINA/PLANCHA                       141
 PARIETAL/RUPESTRE                    112
 TESERA                                66
 LAPIDA                                50
 PESA DE TELAR                         23
 Ánfora                                22
                                       22
 INSTRUMENTO                           20
 PLACA                                 16
 ESTRUC. ARQUIT.                        9
 ESTELA                                 9
 estela                                 8
 Pedestal                               7
 FUSAYOLA                               5
 fusayola                               5
 Monetiforme                            4
 PEDESTAL                               4
 Óstracon                               4
 ARMA                                   3
 Tésera                                 3
 Placa-lingote                          2
 MOSAICO                                2
 ESTATUA                 

In [73]:
data['medium_cat'] = data['soporte'].str.upper()
data['medium_cat'].value_counts()

medium_cat
RECIPIENTE                          1179
LAMINA/PLANCHA                       141
PARIETAL/RUPESTRE                    112
TESERA                                66
LAPIDA                                50
PESA DE TELAR                         23
ÁNFORA                                23
                                      22
INSTRUMENTO                           20
ESTELA                                17
PLACA                                 16
PEDESTAL                              11
FUSAYOLA                              10
ESTRUC. ARQUIT.                        9
ÓSTRACON                               4
MONETIFORME                            4
DOLIUM                                 3
ARMA                                   3
ESTATUA                                3
TÉSERA                                 3
RECIPIENTE DE LUJO                     2
PLACA-LINGOTE                          2
PESO                                   2
COLGANTE                               2
TAPAD

We'll collect all the single entries into a miscelaneous category

In [74]:
medium_cat_count = data['medium_cat'].value_counts()
data.loc[data['medium_cat'].apply(lambda m : medium_cat_count[m]) < 2,'medium_cat'] = 'MISC.'

In [75]:
data['medium_cat'].value_counts(),data['medium_cat'].value_counts().shape

(medium_cat
 RECIPIENTE            1179
 LAMINA/PLANCHA         141
 PARIETAL/RUPESTRE      112
 TESERA                  66
 LAPIDA                  50
 MISC.                   24
 PESA DE TELAR           23
 ÁNFORA                  23
                         22
 INSTRUMENTO             20
 ESTELA                  17
 PLACA                   16
 PEDESTAL                11
 FUSAYOLA                10
 ESTRUC. ARQUIT.          9
 ÓSTRACON                 4
 MONETIFORME              4
 ARMA                     3
 DOLIUM                   3
 ESTATUA                  3
 TÉSERA                   3
 TAPADERA                 2
 COLGANTE                 2
 PESO                     2
 ESCULTURA                2
 MOSAICO                  2
 PLACA-LINGOTE            2
 RECIPIENTE DE LUJO       2
 Name: count, dtype: int64,
 (28,))

In [76]:
medium_categoriser = tf.keras.layers.StringLookup()
medium_categoriser.adapt(data['medium_cat'])
data['medium_cat_code'] = medium_categoriser(data['medium_cat'])

### Writing direction (direccion_escritura)

In [77]:
data['direccion_escritura'].value_counts(),data['direccion_escritura'].value_counts().shape

(direccion_escritura
 DEXTROGIRA     1597
 LEVOGIRA         70
                  67
 VERTICAL         12
 ESPIRAL           7
 BUSTROFEDON       4
 Name: count, dtype: int64,
 (6,))

In [78]:
writing_direction_categoriser = tf.keras.layers.StringLookup()
writing_direction_categoriser.adapt(data['direccion_escritura'])
data['writing_direction_cat_code'] = writing_direction_categoriser(data['direccion_escritura'])

### Technique (tecnica)

In [79]:
data['tecnica'].value_counts(),data['tecnica'].value_counts().shape

(tecnica
 INCISION                   1455
 PINTADO                     143
 IMPRESO                      80
 PUNTILLADO                   30
                              23
 Incisión ante coctionem      12
 REPUJADO                      3
 Incisión ante cocturam        2
 Incisión ante cocción         2
 Grafitado                     1
 esgrafiado                    1
 teselas                       1
 INCISION ante COCCIÓN         1
 mosaico                       1
 Mosaico                       1
 impresión e incisión          1
 Name: count, dtype: int64,
 (16,))

Some entries are specifications of more general ones. Let's keep it generalised.

In [80]:
data['technique_cat'] = data['tecnica'].copy().str.upper()
data.loc[data['technique_cat'].str.contains('INCISI[OÓ]N'),'technique_cat'] = 'INCISION'
data['technique_cat'].value_counts(),data['technique_cat'].value_counts().shape

(technique_cat
 INCISION      1473
 PINTADO        143
 IMPRESO         80
 PUNTILLADO      30
                 23
 REPUJADO         3
 MOSAICO          2
 GRAFITADO        1
 ESGRAFIADO       1
 TESELAS          1
 Name: count, dtype: int64,
 (10,))

In [81]:
technique_categoriser = tf.keras.layers.StringLookup()
technique_categoriser.adapt(data['technique_cat'])
data['technique_cat_code'] = technique_categoriser(data['technique_cat'])

### Signary (signario)

In [82]:
data['signario'].value_counts(),data['signario'].value_counts().shape

(signario
 LEVANTINO                                                   1491
 CELTIBERICO E.                                                57
 MERIDIONAL                                                    54
 CELTIBERICO W.                                                39
 GRECOIBERICO                                                  35
 LATINO                                                        34
 CELTIBERICO                                                   28
                                                                9
 Celtibérico                                                    2
 CELTIBERICO E. / LEVANTINO                                     2
 GRIEGO                                                         2
 Meridional?                                                    1
 GRECOIBERICO ó GRIEGO                                          1
 CELTIBERICO W. ?                                               1
 CELTIBERICO, probablemente oriental (Simón Cornago 2013)       1


We need to do some cleanup here

In [83]:
data['signary_cat'] = data['signario'].copy()
data.loc[data['signary_cat']=='CELTIBERICO E. / LEVANTINO','signary_cat'] = ''
data.loc[data['signary_cat']=='Celtibérico','signary_cat'] = 'CELTIBERICO'
data.loc[data['signary_cat']=='Meridional?','signary_cat'] = ''
# We'll assume they are Celtiberian
data.loc[data['signary_cat']=='CELTIBERICO W. ?','signary_cat'] = 'CELTIBERICO'
data.loc[data['signary_cat']=='CELTIBERICO, probablemente oriental (Simón Cornago 2013)','signary_cat'] = 'CELTIBERICO'
# We'll assume GRECOIBERICO ó GRIEGO is Graecoiberian
data.loc[data['signary_cat']=='GRECOIBERICO ó GRIEGO','signary_cat'] = 'GRECOIBERICO'
data['signary_cat'] = data['signary_cat'].str.upper()
data['signary_cat'].value_counts(),data['signary_cat'].value_counts().shape

(signary_cat
 LEVANTINO         1491
 CELTIBERICO E.      57
 MERIDIONAL          54
 CELTIBERICO W.      39
 GRECOIBERICO        36
 LATINO              34
 CELTIBERICO         32
                     12
 GRIEGO               2
 Name: count, dtype: int64,
 (9,))

In [84]:
signary_categoriser = tf.keras.layers.StringLookup()
signary_categoriser.adapt(data['signary_cat'])
data['signary_cat_code'] = signary_categoriser(data['signary_cat'])

### Dual system (sistema_dual)

In [85]:
data['sistema_dual'].value_counts()

sistema_dual
NO DUAL      1210
DUAL          450
NO APLICA      89
                8
Name: count, dtype: int64

We'll combine the null value with "NO APLICA" (IT DOES NOT APPLY)

In [86]:
data['dual_system_cat'] = data['sistema_dual'].copy()
data.loc[data['dual_system_cat']=='NO APLICA','dual_system_cat'] = ''
data['dual_system_cat'].value_counts()

dual_system_cat
NO DUAL    1210
DUAL        450
             97
Name: count, dtype: int64

In [87]:
dual_system_categoriser = tf.keras.layers.StringLookup()
dual_system_categoriser.adapt(data['dual_system_cat'])
data['dual_system_cat_code'] = dual_system_categoriser(data['dual_system_cat'])

### Separators (separadores)

In [88]:
data['separadores'].value_counts().shape

(80,)

We need to do a cleanup here, too.

In [89]:
data['separators_cat'] = data['separadores'].copy().str.upper().str.replace('\.','',regex=True)

We will call MIXED all those categories that involve several entries, Y (AND), and O (OR)

In [90]:
data.loc[data['separators_cat'].str.contains('\s+[YOÓ]\s+|:'),'separators_cat'] = 'MIXED'

In [91]:
data['separators_cat'].value_counts().index

Index(['CARECE', '', 'DOS PUNTOS', 'TRES PUNTOS', 'MIXED', 'UN PUNTO', 'RAYA',
       'CUATRO PUNTOS', '-', 'CINCO PUNTOS', 'ESPACIO', 'UN PUNTO?',
       'CUADRADO', 'CARECE ?', '2 RAYAS ALTAS', 'UN PUNTO ?', 'CUADRADO?',
       'SEIS PUNTOS', 'CRUZ', 'ROMBOS', 'TRES LÍNEAS CORTAS',
       'DOS PUNTOS/ORNAMENTO', '¿UN PUNTO?', 'DOS PUNTOS CONSERVADOS',
       'OCHO PUNTOS?', 'RAYA ALTA', '2-4 PUNTOS', 'AL MENOS TRES PUNTOS',
       'TRES-CINCO PUNTOS', '9 PUNTOS', '1 PUNTO', '3 RAYAS', 'ESPACIOS',
       'DOS TRIÁNGULOS', 'UN PUNTO (SIMÓN CORNAGO)', 'DOS RAYITAS'],
      dtype='object', name='separators_cat')

The "-" separator is already split into several entries so it does not appear in the texts. We'll put the outlier separators into a MISC. category

In [92]:
MISC_SEPARATORS = ['-','CUADRADO','2 RAYAS ALTAS','CRUZ','ROMBOS','RAYA ALTA','DOS TRIÁNGULOS']
data.loc[data['separators_cat'].str.match('|'.join(MISC_SEPARATORS)),'separators_cat'] = 'MISC.'

In [93]:
data.loc[data['separators_cat']=='UN PUNTO (SIMÓN CORNAGO)','separators_cat'] = 'UN PUNTO'
data.loc[data['separators_cat']=='ESPACIOS','separators_cat'] = 'ESPACIO'

In [94]:
data.loc[data['separators_cat'].str.contains('\?'),'separators_cat'] = 'UNSURE'

In [95]:
data['separators_cat'].value_counts()

separators_cat
CARECE                    1116
                           323
DOS PUNTOS                  93
TRES PUNTOS                 72
MIXED                       38
UN PUNTO                    33
RAYA                        19
MISC.                       16
CUATRO PUNTOS               16
UNSURE                       8
CINCO PUNTOS                 6
ESPACIO                      6
SEIS PUNTOS                  1
TRES LÍNEAS CORTAS           1
DOS PUNTOS/ORNAMENTO         1
DOS PUNTOS CONSERVADOS       1
AL MENOS TRES PUNTOS         1
TRES-CINCO PUNTOS            1
2-4 PUNTOS                   1
9 PUNTOS                     1
1 PUNTO                      1
3 RAYAS                      1
DOS RAYITAS                  1
Name: count, dtype: int64

In [96]:
separators_count = data['separators_cat'].value_counts()
data.loc[data['separators_cat'].apply(lambda s : separators_count[s]) < 2,'separators_cat'] = 'MISC.'

In [97]:
data['separators_cat'].value_counts(),data['separators_cat'].value_counts().shape

(separators_cat
 CARECE           1116
                   323
 DOS PUNTOS         93
 TRES PUNTOS        72
 MIXED              38
 UN PUNTO           33
 MISC.              27
 RAYA               19
 CUATRO PUNTOS      16
 UNSURE              8
 CINCO PUNTOS        6
 ESPACIO             6
 Name: count, dtype: int64,
 (12,))

In [98]:
separators_categoriser = tf.keras.layers.StringLookup()
separators_categoriser.adapt(data['separators_cat'])
data['separators_cat_code'] = separators_categoriser(data['separators_cat'])

# Rename the columns for consistency (English)

In [99]:
data.columns

Index(['yacimiento', 'refMLH', 'refHesperia', 'texto', 'municipio',
       'provincia', 'material', 'soporte', 'direccion_escritura', 'tecnica',
       'signario', 'sistema_dual', 'separadores', 'datacion',
       'municipality_latitude', 'municipality_longitude', 'province_latitude',
       'province_longitude', 'datacion_min', 'datacion_max', 'datacion_mean',
       'datacion_width', 'clean_text', 'material_cat', 'material_cat_code',
       'medium_cat', 'medium_cat_code', 'writing_direction_cat_code',
       'technique_cat', 'technique_cat_code', 'signary_cat',
       'signary_cat_code', 'dual_system_cat', 'dual_system_cat_code',
       'separators_cat', 'separators_cat_code'],
      dtype='object')

In [100]:
NEW_COLUMNS = ['site','refMLH','refHesperia','text','municipality','province','material',
               'medium','writing_direction','technique','signary','dual_system','separators',
               'dating','municipality_latitude', 'municipality_longitude', 'province_latitude',
                'province_longitude', 'dating_min', 'dating_max', 'dating_mean',
                'dating_width', 'clean_text', 'material_cat', 'material_cat_code',
                'medium_cat', 'medium_cat_code', 'writing_direction_cat_code', 
                'technique_cat', 'technique_cat_code', 'signary_cat', 'signary_cat_code',
                'dual_system_cat', 'dual_system_cat_code', 'separators_cat', 'separators_cat_code']
data.columns = NEW_COLUMNS

The only thing left for each experiment is:

- Filter the inscriptions to exclude FALSE or SUSPICIOUS ones
  - Or LATINA/GRAECA ones
- Filter the inscriptions by signary
- Fill NULL values
- Split texts' listings
- ...

In [101]:
data.shape

(1757, 36)

In [102]:
data.to_csv('./out/palaeohispanic_dataset.csv',encoding='utf8',index=False)